# 03a. Azure Public Dataset V2 — Predictive Analysis - Summary

**Authors:** Fajar Laksono

**Methodology:** CRISP-ML(Q) + CAMS DevOps

---

## 0. Table of Contents
1. [Summary](#1-summary)
    - [1.1. Business Question(s)](#1.1.-business-questions)
    - [1.2. Dataset Details](#1.2.-dataset-details)
    - [1.3. Business Goals & Success Criteria](#1.3.-business-goals--success-criteria)
    - [1.4. Key Findings Preview from Literature Review](#1.4.-key-findings-preview-from-literature-review)
    - [1.5. Quality Risk Register](#1.5.-quality-risk-register)

## 1. Summary

**CRISP-ML(Q) Phase:** Business Understanding

### 1.1. Business Question(s)
How can we accurately predict cloud resource utilization, detect wasted resources, forecast costs, and segment workloads to enable data-driven FinOps decisions?

### 1.2. Dataset Details
- **Source:** Azure Public Dataset V2 (2019 VM traces)
- **Size:** ~2.7 million VM records
- **Time range:** 30-day trace
- **Tables:** `vmtable`, `subscriptions`, `deployments`, `azure_pricing`, `cpu_readings`

### 1.3. Business Goals & Success Criteria

| Goal | Task | Success Criteria | Business Impact |
|------|------|-----------------|-----------------|
| Predict CPU utilization | Regression: avg_cpu | MAPE < 15%, R² > 0.75 | Rightsizing recommendations |
| Detect idle VMs | Classification: is_idle | F1 > 0.85 | Shutdown candidates |
| Quantify waste | Regression: waste_fraction | MAPE < 15% | Waste reduction tracking |
| Forecast costs | Regression: vm_cost | R² > 0.70 | Budget planning |
| Segment workloads | Clustering: K-Means | Silhouette > 0.3 | Auto-scaling rules |
| Catch anomalies | Isolation Forest | Business validation | Cost anomaly alerting |
| Timeseries forecast | LSTM/GRU | MAE < 3.0 CPU | Capacity planning |

### 1.4. Key Findings Preview from Literature Review
- XGBoost consistently outperforms other tabular models for regression and classification tasks
- `lifetime_hours`, `max_cpu`, and `memory_per_core` are the strongest predictors across all targets
- K-Means reveals 4 distinct workload patterns with clear business interpretations
- Isolation Forest identifies ~5% anomalous VMs representing disproportionate costs
- Bidirectional GRU provides best timeseries accuracy for CPU forecasting

### 1.5 Quality Risk Register

**Purpose:** Identify potential failure points early in the ML lifecycle, per CRISP-ML(Q) risk-based thinking principle.

| # | Risk | Phase | Likelihood | Impact | Mitigation |
|---|------|-------|-----------|--------|------------|
| R1 | Data drift (2019 patterns vs 2026 usage) | Monitoring | Medium | High | Retrain on newer data when available; monitor metric regression across runs via `run_log.csv` |
| R2 | Missing pricing or subscription data | Data Preparation | Low | Medium | Fallback to NaN; downstream models must handle missing values gracefully |
| R3 | Target leakage via correlated features | Feature Engineering | Medium | High | `get_feature_target_columns()` excludes target-related columns; review SHAP for unexpected feature dominance |
| R4 | Timeseries overfitting (few VMs) | Modeling | High | Medium | Early stopping, limit model complexity, cross-validation per VM |
| R5 | CPU readings memory blowup | Data Preparation | Low | High | Already mitigated via DuckDB out-of-core parquet glob (no `pd.concat`) |
| R6 | Skewed waste_tier distribution | Evaluation | Medium | Low | Use weighted F1 score; apply SMOTE if minority class recall < 0.7 |